In [1]:
!sudo apt update -qq
!sudo apt install -y python3.10 python3.10-venv python3.10-dev -qq

197 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following additional packages will be installed:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3-pip-whl python3-setuptools-whl python3.10-minimal
Suggested packages:
  python3.10-doc binfmt-support
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-dev python3.10-venv
The following packages will be upgraded:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-minimal
6 upgraded, 4 newly installed, 0 to remove and 191 not upgraded.
Need to get 15.2 MB of archives.
After this operation, 3,284 kB of additional disk space will be used.
debconf: unable to initialize frontend: Dialog
debconf: (No us

In [2]:
!git clone https://huggingface.co/spaces/khang119966/DeepSeek-OCR-DEMO
%cd DeepSeek-OCR-DEMO

Cloning into 'DeepSeek-OCR-DEMO'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 57 (delta 0), reused 0 (delta 0), pack-reused 54 (from 1)
Receiving objects: 100% (57/57), 290.30 KiB | 10.75 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/DeepSeek-OCR-DEMO


In [3]:
!python3.10 -m venv /content/DeepSeek-OCR-DEMO/py310

In [4]:
!source ./py310/bin/activate

/bin/bash: line 1: ./py310/bin/activate: No such file or directory


In [ ]:
!/content/DeepSeek-OCR-DEMO/py310/bin/pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu118

In [5]:
%%writefile requirements.txt
# torch==2.6.0
transformers==4.46.3
tokenizers==0.20.3
einops
addict
easydict
gradio>=4.0.0
spaces>=0.20.0
Pillow>=10.0.0
safetensors>=0.4.0
accelerate>=0.24.0
sentencepiece>=0.1.99
protobuf>=3.20.0
# torchvision==0.21.0
flash-attn @ https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu11torch2.6cxx11abiTRUE-cp310-cp310-linux_x86_64.whl


Overwriting requirements.txt


In [ ]:
!/content/DeepSeek-OCR-DEMO/py310/bin/pip install -r requirements.txt

In [7]:
!/content/DeepSeek-OCR-DEMO/py310/bin/pip install pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.2 MB/s eta 0:00:0000:0100:01


In [8]:
%%writefile  run_model.py
# run_model.py
import argparse
import torch
from transformers import AutoModel, AutoTokenizer
from PIL import Image
import os
import fitz  # PyMuPDF - pip install pymupdf
import io
import re
import tempfile
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
OUTPUT_DIR = "./ocr_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ────────────────────────────────────────────────
# Model & Tokenizer loading
# ────────────────────────────────────────────────
print("Loading DeepSeek-OCR model and tokenizer...")

model_name = "deepseek-ai/DeepSeek-OCR"  # or "deepseek-ai/DeepSeek-OCR-2" for the newer variant

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_fast=False,
)
# Fix padding warnings – add this block
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    else:
        # DeepSeek-OCR often lacks both; use a placeholder
        tokenizer.add_special_tokens({'pad_token': '<pad>'})
        tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids('<pad>')
    print(f"Padding fixed: pad_token_id now = {tokenizer.pad_token_id}")
    
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    _attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

model = model.eval()

if torch.cuda.is_available():
    print("Moving model to GPU")
    model = model.cuda()
else:
    print("Running on CPU – will be slow")
# print("Running on CPU – will be slow")
# ────────────────────────────────────────────────
# OCR single page
# ────────────────────────────────────────────────
def ocr_page(image: Image.Image) -> str:
    try:
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
            image.save(tmp_file.name, format="PNG")
            image_path = tmp_file.name

        prompt = "<image>\n<|grounding|>Convert this document page to clean markdown format. " \
                 "Preserve:\n" \
                 "- Headings and subheadings (#, ##, ###)\n" \
                 "- Bold, italic, underline\n" \
                 "- Bullet and numbered lists\n" \
                 "- Tables using proper markdown | syntax\n" \
                 "- Code blocks (```language ... ```)\n" \
                 "- Math equations ($...$ inline, $$...$$ display)\n" \
                 "- Layout and reading order as much as possible\n" \
                 "Output **only** the markdown content, no extra text, no explanations, no ```markdown wrapper."

        res = model.infer(
            tokenizer=tokenizer,
            prompt=prompt,
            image_file=image_path,           # still needed
            output_path=OUTPUT_DIR,          # ← REQUIRED FIX
            base_size=1024,
            image_size=640,
            crop_mode=True,
            save_results=False,              # you can keep False
            test_compress=False,
            # eval_mode=True                 # sometimes needed to actually get text output – try adding if still empty
        )

        os.unlink(image_path)

        response = str(res).strip()
        response = re.sub(r'^```markdown\s*|\s*```$', '', response, flags=re.IGNORECASE | re.MULTILINE).strip()
        response = response.replace("<|endoftext|>", "").strip()
        return response if response else "[Empty output from model]"

    except Exception as e:
        import traceback
        err_msg = f"[OCR ERROR] {str(e)}\n{traceback.format_exc()[:500]}"
        if 'image_path' in locals() and os.path.exists(image_path):
            os.unlink(image_path)
        return err_msg


# ────────────────────────────────────────────────
# PDF → Markdown
# ────────────────────────────────────────────────
def pdf_to_markdown(pdf_path: str, output_md: str = None) -> str:
    if not os.path.isfile(pdf_path):
        return f"File not found: {pdf_path}"

    if output_md is None:
        output_md = os.path.splitext(pdf_path)[0] + ".md"

    print(f"Processing: {pdf_path}")
    print(f"Output: {output_md}")

    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        return f"Cannot open PDF: {str(e)}"

    md_parts = []

    for page_num, page in enumerate(doc, 1):
        print(f"  Page {page_num}/{len(doc)} ...")
        try:
            pix = page.get_pixmap(dpi=180, alpha=False)
            img_bytes = pix.tobytes("png")
            pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

            markdown_text = ocr_page(pil_img)
            md_parts.append(f"# Page {page_num}\n\n{markdown_text}\n\n---\n")

        except Exception as e:
            md_parts.append(f"# Page {page_num}\n\n**OCR failed**\n{str(e)}\n\n---\n")

    full_markdown = "\n".join(md_parts)

    with open(output_md, "w", encoding="utf-8") as f:
        f.write(full_markdown)

    doc.close()
    print(f"Done! Markdown saved to: {output_md}")
    return output_md


# ────────────────────────────────────────────────
# CLI
# ────────────────────────────────────────────────
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="PDF → Markdown using DeepSeek-OCR")
    parser.add_argument("--input_f", type=str, help="Path to input PDF")
    parser.add_argument("-o", "--output", type=str, default=None,
                        help="Output .md path (default: input.pdf → input.md)")
    args = parser.parse_args()

    result = pdf_to_markdown(args.input_f, args.output)

    print("\n" + "═" * 70)
    print("Conversion finished.")
    print(f"Markdown file: {result}")
    print("═" * 70 + "\n")

Writing run_model.py


In [12]:
!/content/DeepSeek-OCR-DEMO/py310/bin/python run_model.py  --input_f "/kaggle/input/datasets/pertersmith9595/tra-cuu/Thong_Tin_Tra_Cuu_MoRong (1).pdf" \
    -o  output.md

Loading DeepSeek-OCR model and tokenizer...
modeling_deepseekv2.py: 82.2kB [00:00, 23.6MB/s]
configuration_deepseek_v2.py: 10.6kB [00:00, 19.4MB/s]
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- configuration_deepseek_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- modeling_deepseekv2.py
- configuration_deepseek_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
deepencoder.py: 38.0kB [00:00, 49.2MB/s]
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downlo